In [655]:
import numpy as np
import pymap3d as pm
import plotly.graph_objects as go
from rotation_matrix import rotation_matrix
np.set_printoptions(suppress=True)

In [663]:
# Observer
a1_lat = 36.7341597
a1_lon = -117.2166654
a1_alt = 11000 * 0.3048
a1_roll = 0
a1_pitch = 2
a1_heading = 0

In [696]:
# Target
rap_lat = 36.7290516
rap_lon = -117.2165206
rap_alt = 11500 * 0.3048
rap_roll = 0
rap_pitch = 4
rap_heading = 355

In [697]:
# Convert coordinates to ENU frame relative to the observers location
az, el, slant_range = pm.geodetic2aer(rap_lat, rap_lon, rap_alt, a1_lat, a1_lon, a1_alt)
print('Azimuth: ', az)
print('Elevation: ', el)
print('Slant range (ft): ', slant_range/0.3048)
print('Ground range (ft): ', np.sqrt(slant_range**2 - np.abs(a1_alt - rap_alt)**2))

Azimuth:  178.69286648285987
Elevation:  15.034099787476736
Slant range (ft):  1927.250641329371
Ground range (ft):  567.3125594964124


In [698]:
az = np.radians(az)
az_rmat = np.array([
    [np.cos(az), -1*np.sin(az), 0],
    [np.sin(az), np.cos(az), 0],
    [0, 0, 1]
])
el = np.radians(el)
el_rmat = np.array([
    [np.cos(el), 0, np.sin(el)],
    [0, 1, 0],
    [-1*np.sin(el), 0, np.cos(el)]
])

In [699]:
a1_rmat = rotation_matrix(a1_roll, a1_pitch, a1_heading, isTarget=False)
rap_rmat = rotation_matrix(rap_roll, rap_pitch, rap_heading, isTarget=True)

In [700]:
rmat = rap_rmat @ a1_rmat @ az_rmat @ el_rmat

In [701]:
look = np.atan2(-1*rmat[1, 0], -1*rmat[0, 0])
look = np.mod(np.round(np.degrees(look), 6), 360)

depr = np.asin(rmat[2, 0])
depr = np.degrees(depr)

twist = np.atan2(-1*rmat[2, 1], rmat[2, 2])
twist = np.degrees(twist)

print(f'Look: {look:.2f}')
print(f'Depression: {depr:.2f}')
print(f'Twist: {twist:.2f}')

Look: 3.77
Depression: -17.03
Twist: -0.32
